<a href="https://colab.research.google.com/github/Mihan0207/Corporate_Bank_Loan_Automation/blob/main/Corporate_Loan_Automation_Database.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Corporate Bank Loan Database**

**Purpose:**
Extracts financial data from Yahoo Finance and company registration data from the Companies House API for companies in the FTSE 100, then generates a structured CSV report.

### Explanation of the `pip install` cell
This cell uses `pip` (Python's package installer) to install several essential Python libraries that will be used throughout the notebook:

*   **`pandas`**: A powerful library for data manipulation and analysis, especially with tabular data.
*   **`yfinance`**: Used to download historical market data from Yahoo Finance.
*   **`requests`**: An elegant and simple HTTP library for making web requests, often used to interact with APIs.
*   **`lxml`**: A parsing library for XML and HTML, known for its speed and efficiency, often used in conjunction with `requests` or web scraping tools like BeautifulSoup.

These libraries are crucial for extracting financial data, company registration details, and then processing and reporting that information.

In [1]:
!pip install pandas yfinance requests lxml

In [2]:
import pandas as pd
import yfinance as yf
import requests
import time
import logging
from urllib.parse import quote
from io import StringIO
import numpy as np
import re

The following cell stores authentication key for **Companies House API** access.

In [3]:
COMPANIES_HOUSE_API_KEY = "44977897-b45d-429f-86a9-c36ba5fcb227"

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)


#**Class: CorporateIntelligence**

Encapsulates all functionality into a reusable object.

 ***__init__() — Setup***

Initializes:

> API key

> Base Companies House API URL

> Authenticated session

> Browser-style headers (for Wikipedia scraping)

In [4]:
class CorporateIntelligence:
    def __init__(self, api_key):
        self.api_key = api_key
        self.ch_api_url = "https://api.company-information.service.gov.uk"
        self.session = requests.Session()
        self.session.auth = (api_key, '')
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        }

#format_telephone()
**Purpose:**

Standardizes phone numbers.

**Logic:**

*  Converts invalid values to "N/A"

*   Removes symbols/spaces
*   Ensures accurate number format



In [5]:
def format_telephone(self, phone):


    if not phone or str(phone).strip() in ['N/A', '0', '', 'None']:
        return 'N/A'

    cleaned = re.sub(r'\D', '', str(phone))

    if len(cleaned) < 7:
        return 'N/A'

    # UK local starting with 0 → convert to 44
    if cleaned.startswith('0'):
        cleaned = '44' + cleaned[1:]

    # UK numbers
    if cleaned.startswith('44'):
        country = '44'
        rest = cleaned[2:]
        return f"+{country} {rest[:4]} {rest[4:]}"

    # US numbers
    if cleaned.startswith('1') and len(cleaned) == 11:
        country = '1'
        rest = cleaned[1:]
        return f"+{country} {rest[:3]} {rest[3:6]} {rest[6:]}"

    # 3-digit country codes (like 682)
    if len(cleaned) >= 10:
        country = cleaned[:3]
        rest = cleaned[3:]
        return f"+{country} {rest[:3]} {rest[3:]}"

    return '+' + cleaned


CorporateIntelligence.format_telephone = format_telephone

#get_primary_contact()
**Purpose:**

Fetches key officer from Companies House by prioritizing roles.

In [6]:
def get_primary_contact(self, company_number):
        """Get primary contact from Companies House officers list"""
        if not company_number or company_number == "N/A":
            return "N/A"

        url = f"{self.ch_api_url}/company/{company_number}/officers"

        try:
            time.sleep(0.6)
            response = self.session.get(url)

            if response.status_code == 200:
                data = response.json()

                priority_roles = [
                    'ceo', 'chief executive',
                    'cfo', 'chief financial',
                    'managing director',
                    'chairman', 'chairwoman', 'chair',
                    'director'
                ]

                officers = []

                for item in data.get('items', []):
                    raw_name = item.get('name', '')

                    corporate_keywords = ['LIMITED', 'LTD', 'PLC', 'CORPORATION', 'CORP', 'INC', 'LLC', 'BUSINESS', 'COMPANY']
                    is_corporate = any(keyword in raw_name.upper() for keyword in corporate_keywords)

                    officer_role = item.get('officer_role', '').lower()
                    if is_corporate or 'corporate' in officer_role or 'nominee' in officer_role:
                        continue

                    if ',' in raw_name:
                        parts = raw_name.split(',')
                        surname = parts[0].strip().title()

                        if len(parts) > 1:
                            forenames = parts[1].strip()

                            title_keywords = ['chief executive officer', 'ceo', 'cfo', 'chief financial officer',
                                            'managing director', 'chairman', 'director', 'secretary']
                            for title in title_keywords:
                                forenames = re.sub(r'\b' + title + r'\b', '', forenames, flags=re.IGNORECASE).strip()

                            forenames = forenames.title()
                            readable_name = f"{forenames} {surname}".strip()
                        else:
                            readable_name = surname
                    else:
                        readable_name = raw_name.title()

                    officers.append({
                        'name': readable_name,
                        'role': officer_role
                    })

                    if len(officers) >= 10:
                        break

                for priority in priority_roles:
                    for officer in officers:
                        if priority in officer['role']:
                            return officer['name']

                if officers:
                    return officers[0]['name']

            return "N/A"

        except Exception as e:
            logger.error(f"Error fetching officers: {e}")
            return "N/A"

CorporateIntelligence.get_primary_contact = get_primary_contact

#get_ftse100_tickers()
**Purpose:**

Scrapes FTSE 100 tickers from Wikipedia.

Cleans ticker format:



*   Replaces "." with "-"
*  Ensures ".L" suffix (London exchange)

And finally returns list.

In [7]:
def get_ftse100_tickers(self):
        """Scrape Wikipedia for FTSE 100 tickers and company names"""
        logger.info("Fetching FTSE 100 list from Wikipedia...")
        url = "https://en.wikipedia.org/wiki/FTSE_100_Index"

        try:
            response = requests.get(url, headers=self.headers)
            tables = pd.read_html(StringIO(response.text))

            df = None
            for table in tables:
                if 'Ticker' in table.columns:
                    df = table
                    break

            if df is None:
                return []

            companies = []
            for _, row in df.iterrows():
                ticker = row['Ticker']
                clean_ticker = ticker.replace(".", "-")
                if not clean_ticker.endswith("L"):
                    clean_ticker = f"{clean_ticker}.L"
                if "BT-A" in clean_ticker:
                    clean_ticker = "BT-A.L"

                dba_name = row['Company']
                companies.append((clean_ticker, dba_name))

            return companies
        except Exception:
            return [("TSCO.L", "Tesco"), ("BP.L", "BP")]

CorporateIntelligence.get_ftse100_tickers = get_ftse100_tickers

#get_financial_deep_dive()
**Purpose:**

Pulls detailed financial data from Yahoo Finance

Extracts:
From **Income Statement:**

1.   Revenue
2.   Cost of Goods Sold
3.   Net Income

**From Balance Sheet:**

1. Total Assets

2. Cash

3. Receivables

4. Inventory

5. PPE (Machinery)

6. Real Estate

7. Payables

8. Short-term Debt

9. Long-term Debt

10. Total Liabilities

Also Extracts:

* Company legal name

*  Sector
*  Website
* Telephone (formatted)

Returns combined dictionary.

In [8]:
def get_financial_deep_dive(self, ticker):

        try:
            stock = yf.Ticker(ticker)
            info = stock.info
            bs = stock.balance_sheet
            incs = stock.income_stmt

            bs_date = "N/A"
            inc_date = "N/A"

            if not bs.empty:
                curr_bs = bs.iloc[:, 0]
                bs_date = curr_bs.name.strftime('%Y-%m-%d')
            else:
                curr_bs = pd.Series(dtype='float64')

            if not incs.empty:
                curr_inc = incs.iloc[:, 0]
                inc_date = curr_inc.name.strftime('%Y-%m-%d')
            else:
                curr_inc = pd.Series(dtype='float64')

            phone = info.get('phone', 'N/A')
            formatted_phone = self.format_telephone(phone)

            data = {
                "Balance Sheet Date": bs_date,
                "Income Statement Date": inc_date,
                "Revenue": curr_inc.get("Total Revenue", 0),
                "Cost of Goods Sold": curr_inc.get("Cost Of Revenue", 0),
                "Net Income": curr_inc.get("Net Income", 0),
                "Total Assets": curr_bs.get("Total Assets", 0),
                "Cash": curr_bs.get("Cash And Cash Equivalents", 0),
                "Accounts Receivable": curr_bs.get("Receivables", curr_bs.get("Accounts Receivable", 0)),
                "Inventory": curr_bs.get("Inventory", 0),
                "Machinery/Equipment": curr_bs.get("Net PPE", 0),
                "Real Estate": curr_bs.get("Properties", curr_bs.get("Land And Improvements", 0)),
                "Accounts Payable": curr_bs.get("Payables", curr_bs.get("Accounts Payable", 0)),
                "Notes Payable (Short Term Debt)": curr_bs.get("Current Debt", 0),
                "Mortgages (Long Term Debt)": curr_bs.get("Long Term Debt", 0),
                "Total Liabilities": curr_bs.get("Total Liabilities Net Minority Interest", 0)
            }

            basic = {
                "Ticker": ticker,
                "Legal Name (Yahoo)": info.get('longName', 'N/A'),
                "Sector": info.get('sector', 'N/A'),
                "Data Source": info.get('website', 'N/A'),
                "Telephone": formatted_phone
            }

            return {**basic, **data}

        except Exception as e:
            return None

CorporateIntelligence.get_financial_deep_dive = get_financial_deep_dive

#search_companies_house()
**Purpose:**

Searches Companies House for official registration data.



Extracts:

1. Company number (Tax ID)

2. Incorporation date

3. Company type

4. Jurisdiction

5. Address

6. Status

Returns structured dictionary.

In [9]:
def search_companies_house(self, company_name):

        if not self.api_key or "YOUR_API_KEY" in self.api_key:
            return {'CH Status': 'API Key Missing', 'Tax ID (Company No)': "N/A"}

        clean_name = company_name.replace(" plc", "").replace(" p.l.c.", "").replace(" PLC", "")
        search_query = quote(clean_name)
        url = f"{self.ch_api_url}/search/companies?q={search_query}&items_per_page=1"

        try:
            time.sleep(0.6)
            response = self.session.get(url)

            if response.status_code == 200:
                data = response.json()
                if data.get('items'):
                    top = data['items'][0]
                    addr = top.get('address', {})
                    company_num = top.get('company_number')

                    raw_jur = top.get('jurisdiction', '')
                    if not raw_jur:
                        if 'scotland' in addr.get('country', '').lower():
                            jur = "Scotland"
                        elif 'northern ireland' in addr.get('country', '').lower():
                            jur = "Northern Ireland"
                        else:
                            jur = "England and Wales"
                    else:
                        jur = raw_jur

                    return {
                        "Tax ID (Company No)": company_num,
                        "Incorporation Date": top.get('date_of_creation', 'N/A'),
                        "Business Structure": top.get('company_type', 'N/A'),
                        "State/Province (Jurisdiction)": jur,
                        "City": addr.get('locality', 'N/A'),
                        "Zip/Post Code": addr.get('postal_code', 'N/A'),
                        "Registered Address": f"{addr.get('address_line_1', '')}, {addr.get('postal_code', '')}",
                        "CH Status": top.get('company_status', 'N/A')
                    }
            return {'CH Status': 'Not Found', 'Tax ID (Company No)': "N/A"}
        except Exception:
            return {'CH Status': 'Error', 'Tax ID (Company No)': "N/A"}

CorporateIntelligence.search_companies_house = search_companies_house

#run_report()
**Main Orchestration Method**


Gets the FTSE 100 list. Then for each company:

Pulls financial data (Yahoo). Searches Companies House. Builds structured row. Finally, stores results in full_report

#Data Cleaning Before Export
**Numeric Columns:**

*   Replace missing values with 0
*   Format with commas (1,000,000)

**Dates:**

*   Truncate to YYYY-MM-DD
*   Prefix with tab (avoids Excel auto-format)



**Company Numbers:**



*   Convert to string
*   Prefix with tab (prevents scientific notation in Excel)




In [10]:
def run_report(self):

        tickers = self.get_ftse100_tickers()
        target_list = tickers
        full_report = []

        print(f"--- Processing {len(target_list)} Companies ---")

        for i, (ticker, dba_name_wiki) in enumerate(target_list):
            print(f"[{i+1}/{len(target_list)}] Processing {dba_name_wiki}...")

            fin = self.get_financial_deep_dive(ticker)
            if not fin:
                continue

            legal_search_name = fin.get('Legal Name (Yahoo)', dba_name_wiki)
            legal = self.search_companies_house(legal_search_name)

            tax_id = legal.get('Tax ID (Company No)')
            conf_score = "100%" if tax_id and tax_id != "N/A" and tax_id != "None" else "0%"

            primary_contact = self.get_primary_contact(tax_id)
            telephone = fin.get('Telephone', 'N/A')

            row = {
                "Ticker": ticker,
                "DBA Name": dba_name_wiki,
                "Legal Name (Yahoo)": fin.get('Legal Name (Yahoo)'),
                "Tax ID (Company No)": tax_id,
                "Business Structure": legal.get('Business Structure'),
                "CH Status": legal.get('CH Status'),
                "Confidence Score (Identity)": conf_score,
                "Primary Contact": primary_contact,
                "Telephone": telephone,
                "Incorporation Date": legal.get('Incorporation Date'),
                "Balance Sheet Date": fin.get('Balance Sheet Date'),
                "Income Statement Date": fin.get('Income Statement Date'),
                "State/Province (Jurisdiction)": legal.get('State/Province (Jurisdiction)'),
                "Registered Address": legal.get('Registered Address'),
                "City": legal.get('City'),
                "Zip/Post Code": legal.get('Zip/Post Code'),
                "Sector": fin.get('Sector'),
                "Data Source": fin.get('Data Source'),
                "Revenue": fin.get('Revenue'),
                "Cost of Goods Sold": fin.get('Cost of Goods Sold'),
                "Net Income": fin.get('Net Income'),
                "Total Assets": fin.get('Total Assets'),
                "Cash": fin.get('Cash'),
                "Accounts Receivable": fin.get('Accounts Receivable'),
                "Inventory": fin.get('Inventory'),
                "Machinery/Equipment": fin.get('Machinery/Equipment'),
                "Real Estate": fin.get('Real Estate'),
                "Accounts Payable": fin.get('Accounts Payable'),
                "Notes Payable (Short Term Debt)": fin.get('Notes Payable (Short Term Debt)'),
                "Mortgages (Long Term Debt)": fin.get('Mortgages (Long Term Debt)'),
                "Total Liabilities": fin.get('Total Liabilities')
            }
            full_report.append(row)

        if full_report:
            df = pd.DataFrame(full_report)

            numeric_cols = [
                "Revenue", "Cost of Goods Sold", "Net Income", "Total Assets",
                "Cash", "Accounts Receivable", "Inventory", "Machinery/Equipment",
                "Real Estate", "Accounts Payable", "Notes Payable (Short Term Debt)",
                "Mortgages (Long Term Debt)", "Total Liabilities"
            ]

            for col in numeric_cols:
                if col in df.columns:
                    df[col] = df[col].fillna(0)

            df = df.fillna("N/A")

            if 'Tax ID (Company No)' in df.columns:
                df['Tax ID (Company No)'] = df['Tax ID (Company No)'].astype(str).apply(
                    lambda x: f"\t{x}" if x and x != "N/A" else "N/A"
                )

            date_cols = ["Incorporation Date", "Balance Sheet Date", "Income Statement Date"]
            for col in date_cols:
                if col in df.columns:
                    df[col] = df[col].astype(str).apply(
                        lambda x: f"\t{x[:10]}" if x and x != "N/A" else "N/A"
                    )

            for col in numeric_cols:
                if col in df.columns:
                    df[col] = df[col].apply(lambda x: f"{x:,.0f}" if isinstance(x, (int, float)) else x)

            filename = "Corporate_Loan_Database.csv"
            df.to_csv(filename, index=False, encoding='utf-8-sig')

            print(f"\nSUCCESS! Processed {len(df)} companies.")
            print(f"Data saved to: {filename}")
        else:
            print("No data extracted.")

CorporateIntelligence.run_report = run_report

#CSV Output
**Corporate_Loan_Database.csv**




*   UTF-8 encoded
*  Clean structured output
*  Ready for Excel / credit modeling

#Entry Point





*  Ensures the script runs only when executed directly (not imported)
*  Creates class instance and runs report



In [11]:
if __name__ == "__main__":
    api = CorporateIntelligence(COMPANIES_HOUSE_API_KEY)
    api.run_report()

--- Processing 100 Companies ---
[1/100] Processing 3i...
[2/100] Processing Admiral Group...
[3/100] Processing Airtel Africa...
[4/100] Processing Alliance Witan...
[5/100] Processing Anglo American plc...
[6/100] Processing Antofagasta plc...
[7/100] Processing Ashtead Group...
[8/100] Processing Associated British Foods...
[9/100] Processing AstraZeneca...
[10/100] Processing Autotrader Group...
[11/100] Processing Aviva...
[12/100] Processing Babcock International...
[13/100] Processing BAE Systems...
[14/100] Processing Barclays...
[15/100] Processing Barratt Redrow...
[16/100] Processing Beazley...
[17/100] Processing Berkeley Group Holdings...
[18/100] Processing BP...
[19/100] Processing British American Tobacco...
[20/100] Processing British Land...
[21/100] Processing BT Group...
[22/100] Processing Bunzl...


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BNZL"}}}


[23/100] Processing Burberry Group...
[24/100] Processing Centrica...
[25/100] Processing Coca-Cola Europacific Partners...
[26/100] Processing Coca-Cola HBC...
[27/100] Processing Compass Group...
[28/100] Processing Convatec...
[29/100] Processing Croda International...
[30/100] Processing DCC plc...
[31/100] Processing Diageo...
[32/100] Processing Diploma...
[33/100] Processing Endeavour Mining...
[34/100] Processing Entain...
[35/100] Processing EasyJet...
[36/100] Processing Experian...
[37/100] Processing F & C Investment Trust...
[38/100] Processing Fresnillo plc...
[39/100] Processing Games Workshop...
[40/100] Processing Glencore...
[41/100] Processing GSK plc...
[42/100] Processing Haleon...
[43/100] Processing Halma plc...
[44/100] Processing Hikma Pharmaceuticals...
[45/100] Processing Hiscox...
[46/100] Processing Howdens Joinery...
[47/100] Processing HSBC...
[48/100] Processing ICG...
[49/100] Processing IHG Hotels & Resorts...
[50/100] Processing IMI...
[51/100] Proces

### Explanation of `yfinance` Error

The `ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BNZL"}}}` indicates that the `yfinance` library was unable to find financial data for the stock ticker 'BNZL' on Yahoo Finance.

This is a common occurrence for several reasons:
*   **Ticker Changes/Delistings**: The company might have changed its ticker symbol or been delisted from the stock exchange.
*   **Data Availability**: Yahoo Finance might not have data for that specific symbol, or there could be a temporary issue with their data service.

The script is designed to handle such errors gracefully; it skips companies for which data cannot be fetched and continues processing the rest of the list.

#What This Script Produces

A structured corporate intelligence dataset combining:

1. Market data

2. Legal registry data

3. Officer information

4. Financial statements

Useful for:

1. Credit analysis

2. Corporate lending

3. Risk assessment

